# Holdout Evaluation

Evaluate the selected edit on holdout prompts and inspect topic-level failure patterns and sample responses.

**Runtime:** T4 GPU

In [ ]:
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

try:
    from google.colab import userdata as colab_userdata
except ImportError:
    colab_userdata = None


def read_secret(name: str) -> str:
    if colab_userdata is not None:
        try:
            return colab_userdata.get(name).strip()
        except Exception:
            pass
    return os.environ.get(name, "").strip()


REPO_URL = "https://github.com/aaliyan1230/false-refusal-suppression.git"
HF_TOKEN = read_secret("HF_TOKEN") or getpass("Enter your HF token (or press Enter to skip): ")
REPO_DIR = Path("/content/false-refusal-suppression")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train,dev]"], check=True)

os.chdir(REPO_DIR)
src_path = REPO_DIR / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(REPO_DIR)
print("HF token loaded:", bool(HF_TOKEN))

In [ ]:
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-4B"
SPLIT_DIR = Path("data/processed/splits/pilot_gemini")
DIRECTION_ARTIFACT = Path("artifacts/directions/qwen3_gemini_pilot_borderline_vs_easy.json")
SEARCH_ARTIFACT = Path("artifacts/edits/qwen3_gemini_pilot_search.json")
EVAL_ARTIFACT = Path("artifacts/eval/qwen3_gemini_pilot_holdout.json")

required_paths = [SPLIT_DIR / "holdout.jsonl", DIRECTION_ARTIFACT, SEARCH_ARTIFACT]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing prebuilt evaluation inputs. Run the earlier pipeline stages first: " + ", ".join(missing)
    )

print(MODEL_ID)
print(EVAL_ARTIFACT)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/run_eval.py",
        "--model-id",
        MODEL_ID,
        "--prompts",
        str(SPLIT_DIR / "holdout.jsonl"),
        "--direction-artifact",
        str(DIRECTION_ARTIFACT),
        "--candidate-json",
        str(SEARCH_ARTIFACT),
        "--output",
        str(EVAL_ARTIFACT),
    ],
    check=True,
 )

In [ ]:
import json
import pandas as pd

with open(EVAL_ARTIFACT, "r", encoding="utf-8") as handle:
    report = json.load(handle)

display(pd.DataFrame([report["metrics"]]))
display(pd.DataFrame.from_dict(report["topic_breakdown"], orient="index").sort_values("refusal_rate", ascending=False))
display(pd.DataFrame(report["response_samples"]))